<a href="https://colab.research.google.com/github/chanita-lab/GE338-Code/blob/main/LAB_3/Lab3_classification_6606614680_%E0%B8%8A%E0%B8%99%E0%B8%B4%E0%B8%95%E0%B8%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install & Import Libraries

!pip install geopandas rasterio folium geemap earthengine-api shapely pyproj fiona rtree scikit-learn -q

import geopandas as gpd
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import numpy as np
import ee
import geemap

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, cohen_kappa_score
from sklearn.model_selection import train_test_split

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-chanita')

In [ ]:
# ROI (Bueng Boraphet)
roi = ee.Geometry.Polygon([
    [
        [100.145, 15.695],
        [100.375, 15.695],
        [100.375, 15.560],
        [100.145, 15.560],
        [100.145, 15.695]
    ]
])


# Sentinel-2 Composite
image = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(roi)
         .filterDate('2024-01-01', '2024-12-31')
         .median()
         .clip(roi))

# RGB Map
Map_rgb = geemap.Map()
Map_rgb.centerObject(roi, 11)

Map_rgb.addLayer(
    image,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000},
    "RGB"
)

Map_rgb.addLayer(roi, {}, "Study Area")

Map_rgb

Map(center=[15.627522525394701, 100.26000000000056], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# Indices
ndvi = image.normalizedDifference(['B8','B4']).rename('NDVI')
ndwi = image.normalizedDifference(['B3','B8']).rename('NDWI')
ndbi = image.normalizedDifference(['B11','B8']).rename('NDBI')

# Feature stack
features = image.select(['B2','B3','B4','B8']).addBands([
    ndvi,
    ndwi,
    ndbi
])

# Map indices
Map_index = geemap.Map()
Map_index.centerObject(roi, 11)

Map_index.addLayer(ndvi, {'min':-1,'max':1,'palette':['blue','white','green']}, 'NDVI')
Map_index.addLayer(ndwi, {'min':-1,'max':1,'palette':['brown','white','blue']}, 'NDWI')
Map_index.addLayer(ndbi, {'min':-1,'max':1,'palette':['green','white','red']}, 'NDBI')

Map_index


Map(center=[15.627522525392026, 100.26000000000218], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# Dynamic World labels
dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
      .filterBounds(roi)
      .filterDate('2024-01-01','2024-12-31')
      .select('label')
      .median()
      .clip(roi))

label = dw.toInt()

# จัด class
#1 Water, ,2 Wetland Vegetation, 3 Agriculture, 4 Forest, 5 Urban
remapped = label.remap(
    [0,3,4,1,6,2,5,7,8],
    [1,2,3,4,5,2,2,2,2]
).rename('class')

In [ ]:
# Sampling
samples = features.addBands(remapped).sample(
    region=roi,
    scale=10,
    numPixels=6000,
    seed=10,
    geometries=True
)

samples = samples.filter(ee.Filter.notNull(['class']))

In [ ]:
# Train/Test split
samples = samples.randomColumn('random')

train = samples.filter(ee.Filter.lt('random',0.8))
test = samples.filter(ee.Filter.gte('random',0.8))

In [ ]:
# Random Forest (50 trees)
rf50 = ee.Classifier.smileRandomForest(
    numberOfTrees=50
).train(
    features=train,
    classProperty='class',
    inputProperties=features.bandNames()
)

rf_test = test.classify(rf50)

rf_matrix = rf_test.errorMatrix('class','classification')

print('RF Confusion Matrix:', rf_matrix.getInfo())
print('RF Overall Accuracy:', rf_matrix.accuracy().getInfo())
print('RF Kappa:', rf_matrix.kappa().getInfo())

producers = [i[0] for i in rf_matrix.producersAccuracy().getInfo()]
users = rf_matrix.consumersAccuracy().getInfo()[0]

print("Producer's Accuracy:", producers)
print("User's Accuracy:", users)

f1_scores = []

for p,u in zip(producers,users):
    if (p+u)==0:
        f1=0
    else:
        f1=2*(p*u)/(p+u)
    f1_scores.append(f1)

print("F1-score per class:",f1_scores)


RF Confusion Matrix: [[0, 0, 0, 0, 0, 0], [0, 41, 17, 12, 3, 1], [0, 23, 78, 116, 21, 4], [0, 8, 33, 645, 2, 7], [0, 10, 20, 31, 14, 3], [0, 2, 10, 68, 5, 39]]
RF Overall Accuracy: 0.6735366859027205
RF Kappa: 0.40746862784039395
Producer's Accuracy: [0, 0.5540540540540541, 0.32231404958677684, 0.9280575539568345, 0.1794871794871795, 0.31451612903225806]
User's Accuracy: [0, 0.4880952380952381, 0.4936708860759494, 0.7396788990825688, 0.3111111111111111, 0.7222222222222222]
F1-score per class: [0, 0.5189873417721519, 0.38999999999999996, 0.8232291001914486, 0.22764227642276422, 0.4382022471910112]


In [ ]:
# Random Forest Map
rf_classified = features.classify(rf50)

Map_rf = geemap.Map()
Map_rf.centerObject(roi,11)

Map_rf.addLayer(
    rf_classified,
    {'min':1,'max':5,'palette':['blue','cyan','yellow','green','red']},
    'Random Forest Classification'
)

# Legend
legend_keys = [
    'Water',
    'Wetland Vegetation',
    'Agriculture',
    'Forest',
    'Urban'
]
legend_colors = [
    (0, 0, 255),    # Blue
    (0, 255, 255),  # Cyan
    (255, 255, 0),  # Yellow
    (0, 128, 0),    # Green
    (255, 0, 0)     # Red
]

Map_rf.add_legend(title='Land Cover Classes', keys=legend_keys, colors=legend_colors)

Map_rf

Map(center=[15.627522525394701, 100.26000000000056], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# Gradient Tree Boost
gtb = ee.Classifier.smileGradientTreeBoost(
    numberOfTrees=100,
    shrinkage=0.1,
    maxNodes=20
).train(
    features=train,
    classProperty='class',
    inputProperties=features.bandNames()
)

gtb_test = test.classify(gtb)

gtb_matrix = gtb_test.errorMatrix('class','classification')

print('GTB Confusion Matrix:',gtb_matrix.getInfo())
print('GTB Overall Accuracy:',gtb_matrix.accuracy().getInfo())
print('GTB Kappa:',gtb_matrix.kappa().getInfo())

producers = [i[0] for i in gtb_matrix.producersAccuracy().getInfo()]
users = gtb_matrix.consumersAccuracy().getInfo()[0]

print("Producer's Accuracy:", producers)
print("User's Accuracy:", users)

f1_scores=[]

for p,u in zip(producers,users):
    if (p+u)==0:
        f1=0
    else:
        f1=2*(p*u)/(p+u)
    f1_scores.append(f1)

print("F1-score per class:",f1_scores)



GTB Confusion Matrix: [[0, 0, 0, 0, 0, 0], [0, 39, 19, 12, 3, 1], [0, 25, 89, 108, 14, 6], [0, 7, 35, 645, 1, 7], [0, 9, 19, 34, 14, 2], [0, 3, 9, 69, 6, 37]]
GTB Overall Accuracy: 0.6793075020610058
GTB Kappa: 0.41821394268410417
Producer's Accuracy: [0, 0.527027027027027, 0.3677685950413223, 0.9280575539568345, 0.1794871794871795, 0.29838709677419356]
User's Accuracy: [0, 0.46987951807228917, 0.52046783625731, 0.7430875576036866, 0.3684210526315789, 0.6981132075471698]
F1-score per class: [0, 0.49681528662420377, 0.43099273607748184, 0.8253358925143953, 0.2413793103448276, 0.4180790960451977]


In [ ]:
# GTB Map
gtb_classified = features.classify(gtb)

Map_gtb = geemap.Map()
Map_gtb.centerObject(roi,11)

Map_gtb.addLayer(
    gtb_classified,
    {'min':1,'max':5,'palette':['blue','cyan','yellow','green','red']},
    'Gradient Tree Boost Classification'
)

Map_gtb.add_legend(title='Land Cover Classes', keys=legend_keys, colors=legend_colors)

Map_gtb

Map(center=[15.627522525394701, 100.26000000000056], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# Export ข้อมูล
rf50_classified = features.classify(rf50)

task_rf = ee.batch.Export.image.toDrive(
    image=rf50_classified,
    description='RF50_Landcover',
    folder='GEE_exports',
    fileNamePrefix='RF50_landcover',
    region=roi,
    scale=10,
    maxPixels=1e13
)

task_rf.start()
print("RF50 export started")

RF50 export started


In [ ]:
gtb_classified = features.classify(gtb)

task_gtb = ee.batch.Export.image.toDrive(
    image=gtb_classified,
    description='GTB_Landcover',
    folder='GEE_exports',
    fileNamePrefix='GTB_landcover',
    region=roi,
    scale=10,
    maxPixels=1e13
)

task_gtb.start()
print("GTB export started")

GTB export started
